# Housing Price Regression Competition

In order to predict a home's sale price, we compared a Ridge regression model with a histogram gradient boosting model and selected gradient boosting because it had a lower validation error.


## 1. Set up and load the data

Place the original `train.csv`, `test.csv`, and `solns.csv` files in the `data`
folder next to this notebook. `train.csv` contains the known sale prices.
`test.csv` contains the homes we need to predict.

In Google Colab, upload the files or mount Google Drive and change `DATA_DIR` to the folder containing them. Run the cells in order.

In [44]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_DIR = Path('data')
OUTPUT_DIR = Path('outputs')

required_files = ['train.csv', 'test.csv', 'solns.csv']
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample = pd.read_csv(DATA_DIR / 'solns.csv')


## 2. Keep a test set separate

We remove Id and SalePrice from the predictors. SalePrice is the target,
while Id identifies each row. We create house age, time since remodeling,
house age squared, log lot area, log living area, a bathroom measure, and
total outdoor area. We also convert quality labels into scores from 1 to 5.

We used 80% of the training data to fit and refine the models and kept 20% for
testing. Both models use the same split, with `random_state=42`, so that we compare them on the same homes.

Here, we measured performance using root mean squared error (RMSE), where RMSE is in dollars, gives more weight to large mistakes, and is better when it is
lower.


In [45]:
def engineer_features(df):
    X = df.drop(columns=["Id", "SalePrice"], errors="ignore").copy()

    X["HouseAge"] = X["YrSold"] - X["YearBuilt"]
    X["YearsSinceRemodel"] = X["YrSold"] - X["YearRemodAdd"]
    X["HouseAgeSquared"] = X["HouseAge"] ** 2

    X["LogLotArea"] = np.log1p(X["LotArea"])
    X["LogLivingArea"] = np.log1p(X["GrLivArea"])
    X["BathroomEquivalent"] = X["FullBath"] + 0.5 * X["HalfBath"]

    X["OutdoorArea"] = X[
        ["WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch"]
    ].sum(axis=1, min_count=1)

    quality_scores = {"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}

    for column in ["ExterQual", "KitchenQual", "HeatingQC"]:
        X[f"{column}Score"] = X[column].map(quality_scores)

    for column in ["MSSubClass", "YrSold", "MoSold"]:
        X[column] = X[column].astype(str)

    return X


X = engineer_features(train)
y = train["SalePrice"]
X_test = engineer_features(test)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=42
)

## 3. Categorizing the Data

For our first model, we drew inspiration from asset pricing methods, which explains why we created features such as house age, years since remodeling, log lot area, log living area, and a bathroom measure (counting a half bathroom as 0.5). The quality labels were also mapped to scores from 1 to 5.

| Group | Inputs |
| --- | --- |
| Land | Log lot area, lot frontage |
| Structure | Log living area, basement area, garage area, bathroom equivalent |
| Quality | Overall quality, exterior quality score, kitchen quality score |
| Depreciation | House age, house age squared, years since remodeling |
| Condition | Overall condition, heating quality score |
| Amenities | Fireplaces, outdoor area, pool area |

We also keep neighborhood, zoning, building class, sale year, and sale month
as categorical predictors and turned each numerical group into a
single summary measure.

In [46]:

factor_blocks = {
    "land": ["LogLotArea", "LotFrontage"],
    "structure": [
        "LogLivingArea", "TotalBsmtSF", "GarageArea", "BathroomEquivalent"
    ],
    "quality": ["OverallQual", "ExterQualScore", "KitchenQualScore"],
    "depreciation": ["HouseAge", "HouseAgeSquared", "YearsSinceRemodel"],
    "condition": ["OverallCond", "HeatingQCScore"],
    "amenities": ["Fireplaces", "OutdoorArea", "PoolArea"]
}

categorical_columns = [
    "Neighborhood", "MSZoning", "MSSubClass", "YrSold", "MoSold"
]


## 4. First model: Ridge regression with grouped features

For each numerical group, we fill missing values with the median and
standardize the variables so that differences in units do not dominate.
PCA creates one component that summarizes variation within the group. We
then standardize that component. PCA uses the predictor values to find its
weights; it does not use sale prices to decide those weights.

For the categorical variables, missing values receive the label Missing.
One-hot encoding creates an indicator for each category. Ridge regression
then uses the six PCA summaries and the category indicators to predict price.
Its penalty discourages very large coefficients, which can help reduce
overfitting. The pipeline keeps the learned preprocessing steps inside each
training fold during cross-validation.


In [47]:
def factor_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("input_scaler", StandardScaler()),
        ("pca", PCA(n_components=1, svd_solver="full")),
        ("factor_scaler", StandardScaler())
    ])


categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    [
        (name, factor_pipeline(), columns)
        for name, columns in factor_blocks.items()
    ]
    + [("categories", categorical_pipeline, categorical_columns)],
    remainder="drop"
)

model = Pipeline([
    ("factors", preprocessor),
    ("regression", Ridge(solver="lsqr", tol=1e-6, max_iter=10000))
])

### Tune the model settings

We try seven values of alpha, which controls the Ridge penalty. Five-fold
cross-validation within the 80% training portion selects the setting with
the lowest average root mean squared error (RMSE). We then evaluate that
fitted model on the separate 20% validation portion. The saved output below selects alpha=10 and reports a validation RMSE of $36,327.14.


In [38]:
search = GridSearchCV(
    model,
    param_grid={
        "regression__alpha": [0.01, 0.1, 1, 10, 100, 1000, 10000]
    },
    scoring="neg_root_mean_squared_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1,
    refit=True,
    error_score="raise"
)

search.fit(X_train, y_train)

holdout_predictions = search.best_estimator_.predict(X_valid)
holdout_rmse = mean_squared_error(y_valid, holdout_predictions) ** 0.5

print(f"Best alpha: {search.best_params_['regression__alpha']:g}")
print(f"RMSE: ${holdout_rmse:,.2f}")

Best alpha: 10
RMSE: $36,327.14


## 5. Fitting the model and recreating the submission file

We fit a new copy of the selected Ridge pipeline on all labeled training
rows and predict prices for the test rows. This uses the engineered features
prepared earlier for the Ridge model. We check the row count, unique IDs,
finite predictions, and positive prices before saving the_winning.csv.

This first export uses the IDs from test.csv and the next step checks whether
those IDs match the competition's submission template.

In [39]:

final_model = clone(search.best_estimator_)
final_model.fit(X, y)

submission = pd.DataFrame({
    "Id": test["Id"].to_numpy(),
    "SalePrice": final_model.predict(X_test)
})

assert len(submission) == len(test)
assert submission["Id"].is_unique
assert np.isfinite(submission["SalePrice"]).all()
assert (submission["SalePrice"] > 0).all()

submission.to_csv("the_winning.csv", index=False)

### Match the competition's required IDs

The checks below found 260 test rows and 260 template rows. The test
IDs started at 1201, while the template IDs started at 0. The two files
therefore had different ID sets.

Our final export used the template IDs and paired them with predictions
by row position. This assumes the template and test files list homes
in corresponding order. Comparing the ID sets does not establish that
correspondence on its own.

In [40]:
official_test = pd.read_csv(DATA_DIR / 'test.csv')
sample = pd.read_csv(DATA_DIR / 'solns.csv')

test_ids = set(official_test["Id"])
sample_ids = set(sample["Id"])

print("Official test rows:", len(official_test))
print("Sample submission rows:", len(sample))

print("\nFirst test IDs:")
print(official_test["Id"].head(10).tolist())

print("\nFirst sample IDs:")
print(sample["Id"].head(10).tolist())

print("\nIDs in sample but not test:", len(sample_ids - test_ids))
print("IDs in test but not sample:", len(test_ids - sample_ids))

print("\nSame ID sets:", test_ids == sample_ids)
print(
    "Same IDs in the same order:",
    official_test["Id"].tolist() == sample["Id"].tolist()
)

Official test rows: 260
Sample submission rows: 260

First test IDs:
[1201, 1202, 1203, 1204, 1205, 1206, 1207, 1208, 1209, 1210]

First sample IDs:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

IDs in sample but not test: 260
IDs in test but not sample: 260

Same ID sets: False
Same IDs in the same order: False


## 6. Second model: histogram gradient boosting

Our second model was `HistGradientBoostingRegressor`. It builds decision trees
in sequence, with each new tree helping reduce errors from the existing model.
This allows it to capture nonlinear patterns and interactions (like the
possibility that an extra bathroom matters differently for small and large homes).

For this model, we used all original predictors after removing `Id` and
`SalePrice`. We treated `MSSubClass` as a category because its numbers identify
building types. The engineered features and PCA summaries from the Ridge model
are not used here.

We filled missing numerical values with medians and missing categorical values
with `Missing`. `OrdinalEncoder` gives each category an integer code, and we marked those columns as categorical for the model. It is important to note that the codes identify groups rather than a numerical ranking. Unseen categories receive `-1`, which this estimator handles as missing. The tree splits did not require the standardization used by our Ridge model.


In [41]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingRegressor

X = train.drop(columns=["Id", "SalePrice"], errors="ignore").copy()
y = train["SalePrice"]

if "MSSubClass" in X.columns:
    X["MSSubClass"] = X["MSSubClass"].astype(str)

numeric_columns = X.select_dtypes(include="number").columns.tolist()
categorical_columns = X.select_dtypes(exclude="number").columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        keep_empty_features=True
    ))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Missing"
    )),
    ("encoder", OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

categorical_mask = (
    [False] * len(numeric_columns)
    + [True] * len(categorical_columns)
)

boosting_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regression", HistGradientBoostingRegressor(
        loss="squared_error",
        categorical_features=categorical_mask,
        learning_rate=0.05,
        max_iter=400,
        early_stopping=False,
        random_state=42
    ))
])

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=42
)

boosting_search = GridSearchCV(
    boosting_pipeline,
    param_grid={
        "regression__max_leaf_nodes": [7, 15],
        "regression__min_samples_leaf": [10, 20],
        "regression__l2_regularization": [1, 10, 100]
    },
    scoring="neg_root_mean_squared_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1,
    refit=True,
    error_score="raise"
)

boosting_search.fit(X_train, y_train)

predictions = boosting_search.best_estimator_.predict(X_valid)

rmse = mean_squared_error(y_valid, predictions) ** 0.5
print(f"New RMSE: ${rmse:,.2f}")

New RMSE: $35,218.33


## 7. Final Export Code

This cell is intended to fit the selected boosting pipeline on all labeled
rows, predict test prices, use the template IDs, and save
the_new_winning.csv. It includes checks for row counts, unique IDs, and
finite positive predictions.

In [42]:
from sklearn.base import clone

official_test = pd.read_csv(DATA_DIR / 'test.csv')
sample = pd.read_csv(DATA_DIR / 'solns.csv')

X_submission = official_test.drop(
    columns=["Id", "SalePrice"], errors="ignore"
).copy()

if "MSSubClass" in X_submission.columns:
    X_submission["MSSubClass"] = (
        X_submission["MSSubClass"].astype(str)
    )

assert set(X_submission.columns) == set(X.columns), (
    "Training and test predictor columns do not match."
)
X_submission = X_submission[X.columns]

final_boosting_model = clone(boosting_search.best_estimator_)
final_boosting_model.fit(X, y)

predicted_prices = final_boosting_model.predict(X_submission)

predictions = pd.DataFrame({
    "Id": sample["Id"].to_numpy(),
    "SalePrice": predicted_prices
})

assert len(predictions) == len(sample) == 260
assert predictions["Id"].is_unique
assert sample["Id"].is_unique
assert set(predictions["Id"]) == set(sample["Id"])

submission = sample[["Id"]].merge(
    predictions,
    on="Id",
    how="left",
    validate="one_to_one",
    sort=False
)

assert np.isfinite(submission["SalePrice"]).all()
assert (submission["SalePrice"] > 0).all()

submission.to_csv("the_new_winning.csv", index=False)

## 8. Comparing the results

These are the validation results saved in our original project notebook:

| Model | Validation RMSE (USD) |
| --- | ---: |
| Ridge regression with PCA summaries | 36,327.14 |
| Histogram gradient boosting | 35,218.33 |

Gradient boosting reduced RMSE by 1,108.81 USD, or about 3.05%, on the same
validation split. We used this as our final model because of the improvement.
The Ridge model's selected `alpha` was 10.

The two approaches changed both estimator and inputs. Gradient boosting
kept the original predictors, while Ridge used selected groups compressed by
PCA. Its better result therefore does not show that the estimator alone caused
the improvement. Another explanation could be that keeping more information and allowing more flexible patterns may both have helped.


In [43]:

comparison = pd.DataFrame({
    'Model': ['Ridge with PCA summaries', 'Histogram gradient boosting'],
    'Validation RMSE ($)': [ridge_rmse, boosting_rmse],
    'Cross-validation RMSE ($)': [-ridge_search.best_score_, -boosting_search.best_score_],
})
print(comparison.to_string(index=False, float_format=lambda value: f'{value:,.2f}'))
print(f'Validation RMSE reduction: ${ridge_rmse - boosting_rmse:,.2f}')
print(f'Percentage reduction: {100 * (ridge_rmse - boosting_rmse) / ridge_rmse:.2f}%')


                      Model  Validation RMSE ($)  Cross-validation RMSE ($)
   Ridge with PCA summaries            36,327.14                  33,653.45
Histogram gradient boosting            35,218.33                  24,595.79
Validation RMSE reduction: $1,108.81
Percentage reduction: 3.05%


## 9. Conclusion and limitations

We selected histogram gradient boosting because its validation RMSE of
35,218.33 USD was lower than the Ridge model's 36,327.14 USD. This suggests that the
more flexible approach, together with keeping the original predictors, worked
better for our validation data. It still made prediction errors, and the
validation result does not guarantee the same performance on new homes.

We used one 80/20 validation split to compare the approaches, so a different
split could change the results. Because that split informed our model choice,
its score is also not a completely independent final evaluation. RMSE puts
extra weight on large dollar errors, including errors on expensive homes. A
useful next step would be to examine residuals and compare both estimators
using the same predictors to understand where the improvement came from.
